In [1]:
#!pip install pandas sqlalchemy oracledb

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('D:/CSDLPT/new_dataset.csv')

In [3]:
df.head()

,order_id,order_date,order_status,customer_id,customer_name,gender,age,customer_segment,country,city,...,brand,stock_quantity,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location
0,ORD-XAJI0,2024-10-14 11:20:05.496679,Completed,CUS-6DPBH,Drew Warren,Male,53,Regular,Netherlands,Utrecht,...,Nike,99,120.51,3,10,36.15,325.38,28.14,Apple Pay,UK
1,ORD-NHJ7X,2024-10-21 00:49:44.681065,Completed,CUS-G0FN9,Victor Sullivan,Male,61,Regular,United States,Los Angeles,...,Asus,146,60.33,2,15,18.10,102.56,8.96,Bank Transfer,UK
2,ORD-YTJXE,2025-03-17 19:49:36.983317,Completed,CUS-Q85JS,Jason Garcia,Male,50,VIP,France,Nice,...,Asus,899,295.89,3,0,0.00,887.67,69.56,Credit Card,UK
3,ORD-EIMVI,2024-09-27 06:24:44.913768,Completed,CUS-CWI64,David Norris,Male,50,Regular,Italy,Milan,...,Adidas,884,186.33,1,0,0.00,186.33,16.78,Credit Card,USA-West
4,ORD-OR56F,2025-05-21 17:10:48.401882,Completed,CUS-O72KZ,Laura Phillips,Female,27,Regular,Germany,Berlin,...,Anker,82,427.16,1,10,42.72,384.44,23.74,Credit Card,UK


In [4]:
# Lọc các cột cho bảng CUSTOMERS và loại bỏ trùng lặp customer_id
customers_df = df[[
    'customer_name', 
    'gender', 
    'age', 
    'country', 
    'city'
]].drop_duplicates()

In [5]:
from sqlalchemy import create_engine

In [6]:
# --- BƯỚC 2: CẤU HÌNH KẾT NỐI ORACLE LOCAL ---
#user connection 
USER = 'branch1'      
PASSWORD = 'branch1'
HOST = 'localhost'
PORT = '1521'
SERVICE_NAME = 'xe'         # Thường là xe hoặc orcl

# Tạo connection string cho SQLAlchemy
# Sử dụng 'oracle+oracledb' để dùng thư viện thế hệ mới
conn_url = f"oracle+oracledb://{USER}:{PASSWORD}@{HOST}:{PORT}/?service_name={SERVICE_NAME}"
engine = create_engine(conn_url)

In [7]:
# --- BƯỚC 3: NẠP DỮ LIỆU ---
try:
    print("Đang nạp dữ liệu vào bảng CUSTOMERS...")
    customers_df.to_sql(
        name='CUSTOMERS', 
        con=engine, 
        if_exists='append', 
        index=False
    )
    print("Thành công! Dữ liệu đã được nạp vào Oracle Local.")
except Exception as e:
    print(f"Lỗi: {e}")

Đang nạp dữ liệu vào bảng CUSTOMERS...
Thành công! Dữ liệu đã được nạp vào Oracle Local.


C:\Users\DINHPHAT\AppData\Local\Temp\ipykernel_8692\471193883.py:4: UserWarning: The provided table name 'CUSTOMERS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  customers_df.to_sql(


In [8]:
# 1. Lọc và làm sạch dữ liệu cho bảng PRODUCTS_INFO
# Dựa trên danh sách cột bạn đã liệt kê ở yêu cầu trước
products_df = df[[
    'product_name', 
    'category', 
    'brand'
]].drop_duplicates()  # Loại bỏ các sản phẩm trùng tên, loại, nhãn hiệu

# 2. Đẩy dữ liệu vào bảng trong Oracle
try:
    print("Đang nạp dữ liệu vào bảng PRODUCTS_INFO...")
    
    products_df.to_sql(
        name='PRODUCTS_INFO', 
        con=engine, 
        if_exists='append', # Dùng 'append' để thêm dữ liệu vào bảng đã có
        index=False
    )
    
    print(f"✅ Thành công! Đã nạp {len(products_df)} sản phẩm vào PRODUCTS_INFO.")
    
except Exception as e:
    print(f"❌ Lỗi khi nạp bảng PRODUCTS_INFO: {e}")

Đang nạp dữ liệu vào bảng PRODUCTS_INFO...
✅ Thành công! Đã nạp 960 sản phẩm vào PRODUCTS_INFO.


C:\Users\DINHPHAT\AppData\Local\Temp\ipykernel_8692\2434615205.py:13: UserWarning: The provided table name 'PRODUCTS_INFO' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  products_df.to_sql(


In [9]:
import pandas as pd
import numpy as np

if len(df) < 960:
    source_df = pd.concat([df] * (960 // len(df) + 1), ignore_index=True)
else:
    source_df = df.copy()

# 2. Chỉ giữ lại 960 dòng đầu tiên và các cột cần thiết
stock_df = source_df.iloc[:960][['stock_quantity', 'unit_price_usd']].copy()

# 3. Tạo product_id từ 1 đến 960
stock_df['product_id'] = range(1, 961)

# 4. Gán branch_id là id chi nhánh của máy
stock_df['branch_id'] = 1


stock_df['stock_quantity'] = stock_df['stock_quantity'].apply(
    lambda x: x if pd.notnull(x) else np.random.randint(10, 501)
)

stock_df['unit_price_usd'] = stock_df['unit_price_usd'].apply(
    lambda x: x if pd.notnull(x) else round(np.random.uniform(5, 1000), 2)
)

stock_df = stock_df[['product_id', 'branch_id', 'stock_quantity', 'unit_price_usd']]

# 7. Nạp vào Oracle
try:
    print("Đang nạp dữ liệu vào PRODUCTS_STOCK cho chi nhánh 1...")
    stock_df.to_sql(
        name='PRODUCTS_STOCK', 
        con=engine, 
        if_exists='append', 
        index=False
    )
    print(f"✅ Thành công! Đã nạp {len(stock_df)} dòng vào PRODUCTS_STOCK (Branch 1).")
except Exception as e:
    print(f"❌ Lỗi: {e}")
    print("Kiểm tra lại: Bảng BRANCHES phải có ID = 1 và PRODUCTS_INFO phải có ID từ 1-960.")

Đang nạp dữ liệu vào PRODUCTS_STOCK cho chi nhánh 1...
✅ Thành công! Đã nạp 960 dòng vào PRODUCTS_STOCK (Branch 1).


C:\Users\DINHPHAT\AppData\Local\Temp\ipykernel_8692\2977255793.py:32: UserWarning: The provided table name 'PRODUCTS_STOCK' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  stock_df.to_sql(


insert employee

In [10]:
df2 = pd.read_csv('employees_data_v2.csv')
df2['hire_date'] = pd.to_datetime(df2['hire_date'])

In [11]:
df2.head()

,first_name,last_name,email,phone_number,hire_date,salary,branch_id
0,Giang,Smith,giang.smith9783@gmail.com,930380154,2021-01-04,2661.81,1
1,Binh,Nguyen,binh.nguyen6235@gmail.com,363677323,2021-01-06,2712.42,1
2,Viet,Williams,viet.williams1963@gmail.com,930567937,2021-01-12,1562.56,1
3,Son,Le,son.le3805@gmail.com,984303057,2021-01-12,3272.14,1
4,John,Le,john.le1116@gmail.com,736875308,2021-01-26,1754.83,1


In [12]:
# Xóa khoảng trắng thừa ở đầu và cuối tên cột
df2.columns = df2.columns.str.strip()

# Bây giờ chạy lại dòng này sẽ hết lỗi
df2['phone_number'] = '0' + df2['phone_number'].astype(str)

In [13]:
from sqlalchemy import create_engine, types

In [14]:
print("Đang nạp 350 nhân viên")
df2.to_sql(
    name='EMPLOYEES', 
    con=engine,
    if_exists='append', 
    index=False,
    dtype={
        'hire_date': types.Date(),
        'phone_number': types.VARCHAR(20)
    }
)
print("✅ Thành công!")

Đang nạp 350 nhân viên
✅ Thành công!


C:\Users\DINHPHAT\AppData\Local\Temp\ipykernel_8692\1668670242.py:2: UserWarning: The provided table name 'EMPLOYEES' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df2.to_sql(


table order

In [15]:
# 3. Đọc dữ liệu từ file CSV gốc
df_orders_raw = pd.read_csv('D:/CSDLPT/df_part_1.csv')
df_orders_raw.columns = df_orders_raw.columns.str.strip() # Xóa khoảng trắng tên cột


In [16]:
df_orders_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 333374 entries, 0 to 333373
Data columns (total 25 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  333374 non-null  str    
 1   order_date                333374 non-null  str    
 2   order_status              333374 non-null  str    
 3   customer_id               333374 non-null  str    
 4   customer_name             333374 non-null  str    
 5   gender                    333374 non-null  str    
 6   age                       333374 non-null  int64  
 7   customer_segment          333374 non-null  str    
 8   country                   333374 non-null  str    
 9   city                      333374 non-null  str    
 10  customer_loyalty_score    333374 non-null  float64
 11  total_orders_by_customer  333374 non-null  int64  
 12  product_id                333374 non-null  str    
 13  product_name              333374 non-null  str    
 14 

In [17]:
df_orders_raw.head()

,order_id,order_date,order_status,customer_id,customer_name,gender,age,customer_segment,country,city,...,brand,stock_quantity,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location
0,ORD-XAJI0,2024-10-14 11:20:05.496679,Completed,CUS-6DPBH,Drew Warren,Male,53,Regular,Netherlands,Utrecht,...,Nike,99,120.51,3,10,36.15,325.38,28.14,Apple Pay,UK
1,ORD-NHJ7X,2024-10-21 00:49:44.681065,Completed,CUS-G0FN9,Victor Sullivan,Male,61,Regular,United States,Los Angeles,...,Asus,146,60.33,2,15,18.10,102.56,8.96,Bank Transfer,UK
2,ORD-YTJXE,2025-03-17 19:49:36.983317,Completed,CUS-Q85JS,Jason Garcia,Male,50,VIP,France,Nice,...,Asus,899,295.89,3,0,0.00,887.67,69.56,Credit Card,UK
3,ORD-EIMVI,2024-09-27 06:24:44.913768,Completed,CUS-CWI64,David Norris,Male,50,Regular,Italy,Milan,...,Adidas,884,186.33,1,0,0.00,186.33,16.78,Credit Card,USA-West
4,ORD-OR56F,2025-05-21 17:10:48.401882,Completed,CUS-O72KZ,Laura Phillips,Female,27,Regular,Germany,Berlin,...,Anker,82,427.16,1,10,42.72,384.44,23.74,Credit Card,UK


In [18]:
# 1. Lấy dữ liệu tra cứu từ Oracle (Lấy thêm Age và Gender để khớp)
cust_lookup = pd.read_sql("SELECT customer_id, customer_name, age, gender FROM CUSTOMERS", engine)
cust_lookup.columns = cust_lookup.columns.str.lower()
# Làm sạch dữ liệu tra cứu
cust_lookup['customer_name'] = cust_lookup['customer_name'].astype(str).str.strip()
cust_lookup['gender'] = cust_lookup['gender'].astype(str).str.strip()
cust_lookup['age'] = cust_lookup['age'].astype(float)

In [19]:
df_orders_raw.columns = df_orders_raw.columns.str.strip().str.lower()
df_orders_raw['customer_name'] = df_orders_raw['customer_name'].astype(str).str.strip()
df_orders_raw['gender'] = df_orders_raw['gender'].astype(str).str.strip()
df_orders_raw['age'] = df_orders_raw['age'].astype(float)

In [20]:
# 3. Thực hiện Merge trên NHIỀU CỘT để đảm bảo duy nhất
# Việc map thêm age và gender sẽ giúp loại bỏ trường hợp trùng tên nhưng khác người
df_orders = pd.merge(
    df_orders_raw, 
    cust_lookup, 
    on=['customer_name', 'age', 'gender'], 
    how='inner'
)

In [21]:
# KIỂM TRA: Nếu merge xong mà không có dữ liệu, báo lỗi ngay
if df_orders.empty:
    print("❌ LỖI: Không tìm thấy khách hàng nào khớp giữa CSV và Database!")
    print("Vui lòng kiểm tra lại cột 'customer_name' ở cả 2 nơi.")
else:
    print("OK")

OK


In [22]:
df_orders.drop_duplicates(['order_id', 'order_date', 'product_id'],inplace=True)

In [23]:
df_orders.head(20)

,order_id,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,stock_quantity,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id_y
0,ORD-XAJI0,2024-10-14 11:20:05.496679,Completed,CUS-6DPBH,Drew Warren,Male,53.0,Regular,Netherlands,Utrecht,...,99,120.51,3,10,36.15,325.38,28.14,Apple Pay,UK,1
1,ORD-NHJ7X,2024-10-21 00:49:44.681065,Completed,CUS-G0FN9,Victor Sullivan,Male,61.0,Regular,United States,Los Angeles,...,146,60.33,2,15,18.10,102.56,8.96,Bank Transfer,UK,2
2,ORD-YTJXE,2025-03-17 19:49:36.983317,Completed,CUS-Q85JS,Jason Garcia,Male,50.0,VIP,France,Nice,...,899,295.89,3,0,0.00,887.67,69.56,Credit Card,UK,3
3,ORD-EIMVI,2024-09-27 06:24:44.913768,Completed,CUS-CWI64,David Norris,Male,50.0,Regular,Italy,Milan,...,884,186.33,1,0,0.00,186.33,16.78,Credit Card,USA-West,4
5,ORD-OR56F,2025-05-21 17:10:48.401882,Completed,CUS-O72KZ,Laura Phillips,Female,27.0,Regular,Germany,Berlin,...,82,427.16,1,10,42.72,384.44,23.74,Credit Card,UK,5
6,ORD-CHJ75,2024-03-17 16:10:10.356632,Returned,CUS-KPTK9,James Campbell,Male,60.0,Premium,France,Paris,...,517,156.79,1,0,0.00,156.79,10.01,Credit Card,USA-West,6
7,ORD-N78BM,2026-01-13 12:59:24.951221,Completed,CUS-YMU59,Jeffery Summers,Male,33.0,Premium,Canada,Calgary,...,227,75.88,4,5,15.18,288.34,26.98,PayPal,USA-East,7
9,ORD-DP0LV,2024-10-23 14:22:54.319461,Completed,CUS-NUJZA,Justin Brewer,Male,41.0,Regular,Australia,Adelaide,...,693,229.48,2,10,45.90,413.06,31.97,Credit Card,UK,8
10,ORD-D8IL3,2026-01-27 09:22:29.070387,Completed,CUS-B9W1Q,Zachary Wright,Male,63.0,Premium,Italy,Turin,...,467,75.34,4,10,30.14,271.22,21.64,Bank Transfer,Asia-Pacific,9
11,ORD-EIYZC,2024-05-12 06:39:48.681296,Completed,CUS-TOHP6,Bill Moore,Male,72.0,Premium,Belgium,Bruges,...,224,96.42,4,0,0.00,385.68,32.38,Bank Transfer,UK,10


In [24]:
# 4. Kiểm tra số lượng dòng sau khi merge
print(f"Số dòng ban đầu: {len(df_orders_raw)}")
print(f"Số dòng sau khi merge đa điều kiện: {len(df_orders)}")

Số dòng ban đầu: 333374
Số dòng sau khi merge đa điều kiện: 333374


In [25]:
# Gán các giá trị mặc định
df_orders['branch_id'] = 1

# Giả sử ID nhân viên của bạn từ 1 đến 100 
df_orders['employee_id'] = np.random.randint(1, 101, size=len(df_orders))

# Chuyển đổi ngày tháng (đảm bảo đúng cột 'order_date')
if 'order_date' in df_orders.columns:
    df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
    df_orders = df_orders.sort_values(by='order_date').reset_index(drop=True)

In [26]:
if 'customer_id_y' in df_orders.columns:
    df_orders = df_orders.rename(columns={'customer_id_y': 'customer_id'})

In [27]:
df_orders.head()

,order_id,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id,branch_id,employee_id
0,ORD-5A2JR,2024-02-03 04:31:57.621140,Completed,CUS-PLY8L,Curtis Humphrey,Male,46.0,Premium,Germany,Berlin,...,2,15,87.58,496.32,45.38,PayPal,UK,194853,1,44
1,ORD-MI6OD,2024-02-03 04:35:56.872135,Pending,CUS-T4DSS,Robert Jones,Male,42.0,Regular,Belgium,Antwerp,...,2,0,0.00,53.10,3.16,Bank Transfer,USA-West,306047,1,54
2,ORD-EWLM6,2024-02-03 04:36:20.378297,Pending,CUS-A2A2T,Courtney Herman,Female,38.0,Regular,United Kingdom,Leeds,...,3,5,24.82,471.62,33.83,Apple Pay,USA-East,70711,1,14
3,ORD-QXIF2,2024-02-03 04:37:36.076123,Completed,CUS-EX7LC,Jill Shaffer,Female,62.0,Premium,Belgium,Antwerp,...,2,5,16.22,308.22,24.11,Apple Pay,EU-Central,14701,1,50
4,ORD-AD1GA,2024-02-03 04:37:50.848316,Completed,CUS-XKVBG,James Moss,Male,50.0,Regular,United Kingdom,Glasgow,...,2,15,74.59,422.65,35.78,Bank Transfer,Asia-Pacific,24253,1,61


In [28]:
# Chỉ lấy các cột cần thiết cho bảng ORDERS
# Lưu ý: Cột ID từ cust_lookup chắc chắn là 'customer_id'tự thêm yếu số phân biệt .lower() ở trên
cols_to_keep = [
    'order_date', 'customer_id', 'employee_id', 
    'branch_id', 'payment_method', 'tax_usd', 'total_price_usd'
]

In [29]:
df_orders[cols_to_keep].info()

<class 'pandas.DataFrame'>
RangeIndex: 333374 entries, 0 to 333373
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   order_date       333374 non-null  datetime64[us]
 1   customer_id      333374 non-null  int64         
 2   employee_id      333374 non-null  int32         
 3   branch_id        333374 non-null  int64         
 4   payment_method   333374 non-null  str           
 5   tax_usd          333374 non-null  float64       
 6   total_price_usd  333374 non-null  float64       
dtypes: datetime64[us](1), float64(2), int32(1), int64(2), str(1)
memory usage: 19.7 MB


In [30]:
final_orders = df_orders[cols_to_keep]

In [31]:
# Nạp vào Oracle
final_orders.to_sql('ORDERS', con=engine, if_exists='append', index=False)
print(f"✅ Thành công! Đã nạp {len(final_orders)} đơn hàng.")


✅ Thành công! Đã nạp 333374 đơn hàng.


C:\Users\DINHPHAT\AppData\Local\Temp\ipykernel_8692\1215381642.py:2: UserWarning: The provided table name 'ORDERS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  final_orders.to_sql('ORDERS', con=engine, if_exists='append', index=False)


insert order_items

In [32]:
import pandas as pd

# 1. Lấy bảng tra cứu từ Database
# Cần Order_id (mới sinh) và một mã tham chiếu
orders_db = pd.read_sql("SELECT order_id, order_date, customer_id FROM ORDERS", engine)
# Cần Product_id và giá bán
# Lấy Product_id và giá bán tại chi nhánh 1
query = """
    SELECT pi.product_id, pi.product_name, ps.unit_price_usd 
    FROM PRODUCTS_INFO pi
    JOIN PRODUCTS_STOCK ps ON pi.product_id = ps.product_id
    WHERE ps.branch_id = 1
"""
products_db = pd.read_sql(query, engine)
products_db.columns = products_db.columns.str.lower()

In [33]:
# 3. MAPPING (Đây là bước quan trọng nhất)
# Bước A: Map để lấy Product_i
df_items = pd.merge(df_orders_raw, products_db, on='product_name', how='inner')


In [34]:
df_items.drop_duplicates(['order_id', 'order_date', 'product_id_x'],inplace=True)

In [35]:
df_items.info() 

<class 'pandas.DataFrame'>
RangeIndex: 333374 entries, 0 to 6667460
Data columns (total 27 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  333374 non-null  str    
 1   order_date                333374 non-null  str    
 2   order_status              333374 non-null  str    
 3   customer_id               333374 non-null  str    
 4   customer_name             333374 non-null  str    
 5   gender                    333374 non-null  str    
 6   age                       333374 non-null  float64
 7   customer_segment          333374 non-null  str    
 8   country                   333374 non-null  str    
 9   city                      333374 non-null  str    
 10  customer_loyalty_score    333374 non-null  float64
 11  total_orders_by_customer  333374 non-null  int64  
 12  product_id_x              333374 non-null  str    
 13  product_name              333374 non-null  str    
 14

In [36]:
df_items.head()

,order_id,order_date,order_status,customer_id,customer_name,gender,age,customer_segment,country,city,...,unit_price_usd_x,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,product_id_y,unit_price_usd_y
0,ORD-XAJI0,2024-10-14 11:20:05.496679,Completed,CUS-6DPBH,Drew Warren,Male,53.0,Regular,Netherlands,Utrecht,...,120.51,3,10,36.15,325.38,28.14,Apple Pay,UK,1,120.51
20,ORD-NHJ7X,2024-10-21 00:49:44.681065,Completed,CUS-G0FN9,Victor Sullivan,Male,61.0,Regular,United States,Los Angeles,...,60.33,2,15,18.10,102.56,8.96,Bank Transfer,UK,2,60.33
40,ORD-YTJXE,2025-03-17 19:49:36.983317,Completed,CUS-Q85JS,Jason Garcia,Male,50.0,VIP,France,Nice,...,295.89,3,0,0.00,887.67,69.56,Credit Card,UK,3,295.89
60,ORD-EIMVI,2024-09-27 06:24:44.913768,Completed,CUS-CWI64,David Norris,Male,50.0,Regular,Italy,Milan,...,186.33,1,0,0.00,186.33,16.78,Credit Card,USA-West,4,186.33
80,ORD-OR56F,2025-05-21 17:10:48.401882,Completed,CUS-O72KZ,Laura Phillips,Female,27.0,Regular,Germany,Berlin,...,427.16,1,10,42.72,384.44,23.74,Credit Card,UK,5,427.16


In [37]:
# 1. Truy vấn lấy bảng khách hàng từ Oracle
query_cust = "SELECT customer_id, customer_name, age, gender FROM CUSTOMERS"
cust_lookup = pd.read_sql(query_cust, engine)

# 2. Chuẩn hóa tên cột về chữ thường để đồng bộ với code xử lý
cust_lookup.columns = cust_lookup.columns.str.lower()

# 3. (Tùy chọn) Làm sạch dữ liệu để merge chính xác hơn
cust_lookup['customer_name'] = cust_lookup['customer_name'].astype(str).str.strip()
cust_lookup['gender'] = cust_lookup['gender'].astype(str).str.strip()
cust_lookup['age'] = pd.to_numeric(cust_lookup['age'], errors='coerce').fillna(0).astype(int)
df_items_with_cus_id = pd.merge(
    df_orders_raw, 
    cust_lookup[['customer_id', 'customer_name', 'age', 'gender']], 
    on=['customer_name', 'age', 'gender'], 
    how='inner'
)
if 'customer_id_y' in df_items_with_cus_id.columns:
    df_items_with_cus_id = df_items_with_cus_id.rename(columns={'customer_id_y': 'customer_id'})

In [38]:
df_items_with_cus_id.drop_duplicates(['order_id', 'order_date', 'product_id'], inplace=True)

In [39]:
df_items_with_cus_id.info()

<class 'pandas.DataFrame'>
Index: 333374 entries, 0 to 418948
Data columns (total 26 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  333374 non-null  str    
 1   order_date                333374 non-null  str    
 2   order_status              333374 non-null  str    
 3   customer_id_x             333374 non-null  str    
 4   customer_name             333374 non-null  str    
 5   gender                    333374 non-null  str    
 6   age                       333374 non-null  float64
 7   customer_segment          333374 non-null  str    
 8   country                   333374 non-null  str    
 9   city                      333374 non-null  str    
 10  customer_loyalty_score    333374 non-null  float64
 11  total_orders_by_customer  333374 non-null  int64  
 12  product_id                333374 non-null  str    
 13  product_name              333374 non-null  str    
 14  cate

In [40]:
df_items_with_cus_id.head()


,order_id,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,stock_quantity,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id
0,ORD-XAJI0,2024-10-14 11:20:05.496679,Completed,CUS-6DPBH,Drew Warren,Male,53.0,Regular,Netherlands,Utrecht,...,99,120.51,3,10,36.15,325.38,28.14,Apple Pay,UK,1
1,ORD-NHJ7X,2024-10-21 00:49:44.681065,Completed,CUS-G0FN9,Victor Sullivan,Male,61.0,Regular,United States,Los Angeles,...,146,60.33,2,15,18.10,102.56,8.96,Bank Transfer,UK,2
2,ORD-YTJXE,2025-03-17 19:49:36.983317,Completed,CUS-Q85JS,Jason Garcia,Male,50.0,VIP,France,Nice,...,899,295.89,3,0,0.00,887.67,69.56,Credit Card,UK,3
3,ORD-EIMVI,2024-09-27 06:24:44.913768,Completed,CUS-CWI64,David Norris,Male,50.0,Regular,Italy,Milan,...,884,186.33,1,0,0.00,186.33,16.78,Credit Card,USA-West,4
5,ORD-OR56F,2025-05-21 17:10:48.401882,Completed,CUS-O72KZ,Laura Phillips,Female,27.0,Regular,Germany,Berlin,...,82,427.16,1,10,42.72,384.44,23.74,Credit Card,UK,5


In [41]:
df_items_with_cus_id['order_date'] = pd.to_datetime(df_items_with_cus_id['order_date']).dt.floor('s')
orders_db['order_date'] = pd.to_datetime(orders_db['order_date']).dt.floor('s')
df_ready = pd.merge(
    df_items_with_cus_id,
    orders_db[['order_id', 'customer_id', 'order_date']],
    on=['customer_id', 'order_date'],
    how='inner'
)

print(f"Số dòng khớp được ID mới: {len(df_ready)}")

Số dòng khớp được ID mới: 333374


In [42]:

if 'order_id_y' in df_ready.columns:
    df_ready = df_ready.rename(columns={'order_id_y': 'order_id'})

In [43]:
df_ready.info()

<class 'pandas.DataFrame'>
RangeIndex: 333374 entries, 0 to 333373
Data columns (total 27 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   order_id_x                333374 non-null  str           
 1   order_date                333374 non-null  datetime64[us]
 2   order_status              333374 non-null  str           
 3   customer_id_x             333374 non-null  str           
 4   customer_name             333374 non-null  str           
 5   gender                    333374 non-null  str           
 6   age                       333374 non-null  float64       
 7   customer_segment          333374 non-null  str           
 8   country                   333374 non-null  str           
 9   city                      333374 non-null  str           
 10  customer_loyalty_score    333374 non-null  float64       
 11  total_orders_by_customer  333374 non-null  int64         
 12  product_id   

In [44]:
df_ready.head()

,order_id_x,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,unit_price_usd,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id,order_id
0,ORD-XAJI0,2024-10-14 11:20:05,Completed,CUS-6DPBH,Drew Warren,Male,53.0,Regular,Netherlands,Utrecht,...,120.51,3,10,36.15,325.38,28.14,Apple Pay,UK,1,116553
1,ORD-NHJ7X,2024-10-21 00:49:44,Completed,CUS-G0FN9,Victor Sullivan,Male,61.0,Regular,United States,Los Angeles,...,60.33,2,15,18.10,102.56,8.96,Bank Transfer,UK,2,119564
2,ORD-YTJXE,2025-03-17 19:49:36,Completed,CUS-Q85JS,Jason Garcia,Male,50.0,VIP,France,Nice,...,295.89,3,0,0.00,887.67,69.56,Credit Card,UK,3,186900
3,ORD-EIMVI,2024-09-27 06:24:44,Completed,CUS-CWI64,David Norris,Male,50.0,Regular,Italy,Milan,...,186.33,1,0,0.00,186.33,16.78,Credit Card,USA-West,4,108778
4,ORD-OR56F,2025-05-21 17:10:48,Completed,CUS-O72KZ,Laura Phillips,Female,27.0,Regular,Germany,Berlin,...,427.16,1,10,42.72,384.44,23.74,Credit Card,UK,5,216296


In [45]:
df_final_items = pd.merge(
    df_ready, 
    products_db[['product_id', 'product_name']], 
    on='product_name', 
    how='inner'
)

In [46]:
if 'product_id_y' in df_final_items.columns:
    df_final_items = df_final_items.rename(columns={'product_id_y': 'product_id'})

In [47]:
df_final_items.drop_duplicates(['order_id', 'order_date', 'product_id_x'],inplace=True)

In [48]:
df_final_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 333374 entries, 0 to 6667460
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   order_id_x                333374 non-null  str           
 1   order_date                333374 non-null  datetime64[us]
 2   order_status              333374 non-null  str           
 3   customer_id_x             333374 non-null  str           
 4   customer_name             333374 non-null  str           
 5   gender                    333374 non-null  str           
 6   age                       333374 non-null  float64       
 7   customer_segment          333374 non-null  str           
 8   country                   333374 non-null  str           
 9   city                      333374 non-null  str           
 10  customer_loyalty_score    333374 non-null  float64       
 11  total_orders_by_customer  333374 non-null  int64         
 12  product_id_x

In [49]:
df_final_items.head()

,order_id_x,order_date,order_status,customer_id_x,customer_name,gender,age,customer_segment,country,city,...,quantity,discount_percent,discount_amount_usd,total_price_usd,tax_usd,payment_method,warehouse_location,customer_id,order_id,product_id
0,ORD-XAJI0,2024-10-14 11:20:05,Completed,CUS-6DPBH,Drew Warren,Male,53.0,Regular,Netherlands,Utrecht,...,3,10,36.15,325.38,28.14,Apple Pay,UK,1,116553,1
20,ORD-NHJ7X,2024-10-21 00:49:44,Completed,CUS-G0FN9,Victor Sullivan,Male,61.0,Regular,United States,Los Angeles,...,2,15,18.10,102.56,8.96,Bank Transfer,UK,2,119564,2
40,ORD-YTJXE,2025-03-17 19:49:36,Completed,CUS-Q85JS,Jason Garcia,Male,50.0,VIP,France,Nice,...,3,0,0.00,887.67,69.56,Credit Card,UK,3,186900,3
60,ORD-EIMVI,2024-09-27 06:24:44,Completed,CUS-CWI64,David Norris,Male,50.0,Regular,Italy,Milan,...,1,0,0.00,186.33,16.78,Credit Card,USA-West,4,108778,4
80,ORD-OR56F,2025-05-21 17:10:48,Completed,CUS-O72KZ,Laura Phillips,Female,27.0,Regular,Germany,Berlin,...,1,10,42.72,384.44,23.74,Credit Card,UK,5,216296,5


In [50]:
# 3. Ép kiểu dữ liệu lần cuối để chắc chắn không còn "chuỗi" trong các cột số
# Oracle rất nghiêm ngặt với kiểu NUMBER
for col in ['order_id', 'product_id', 'quantity']:
    df_final_items[col] = pd.to_numeric(df_final_items[col], errors='coerce').astype(int)

for col in ['discount_percent', 'discount_amount_usd', 'unit_price_usd', 'total_price_usd']:
    df_final_items[col] = pd.to_numeric(df_final_items[col], errors='coerce').astype(float)


In [51]:
# 4. TÍNH TOÁN CÁC CỘT GIÁ TRỊ (Nếu CSV không có sẵn)
#df_final_items['unit_price_usd'] = df_final_items['unit_price_usd'] # Lấy giá từ DB
df_final_items['discount_amount_usd'] = df_final_items['quantity'] * df_final_items['unit_price_usd'] * (df_final_items['discount_percent'] / 100)
df_final_items['total_price_usd'] = (df_final_items['quantity'] * df_final_items['unit_price_usd']) - df_final_items['discount_amount_usd']

# 5. Lọc cột và nạp vào Oracle
cols_to_load = [
    'order_id', 'product_id', 'quantity', 
    'discount_percent', 'discount_amount_usd', 
    'unit_price_usd', 'total_price_usd'
]


In [52]:
df_final_items.drop_duplicates(['order_id', 'order_date', 'product_id_x'],inplace=True)

In [53]:
df_final_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 333374 entries, 0 to 6667460
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   order_id_x                333374 non-null  str           
 1   order_date                333374 non-null  datetime64[us]
 2   order_status              333374 non-null  str           
 3   customer_id_x             333374 non-null  str           
 4   customer_name             333374 non-null  str           
 5   gender                    333374 non-null  str           
 6   age                       333374 non-null  float64       
 7   customer_segment          333374 non-null  str           
 8   country                   333374 non-null  str           
 9   city                      333374 non-null  str           
 10  customer_loyalty_score    333374 non-null  float64       
 11  total_orders_by_customer  333374 non-null  int64         
 12  product_id_x

In [54]:
# 5. Nạp vào Oracle
try:
    print(f"Đang nạp {len(df_final_items)} dòng vào bảng ORDER_ITEMS...")
    df_final_items[cols_to_load].to_sql(
        'ORDER_ITEMS', 
        con=engine, 
        if_exists='append', 
        index=False, 
        chunksize=5000
    )
    print("✅ Nạp dữ liệu thành công!")
except Exception as e:
    print(f"❌ Lỗi khi nạp: {e}")

Đang nạp 333374 dòng vào bảng ORDER_ITEMS...
✅ Nạp dữ liệu thành công!


C:\Users\DINHPHAT\AppData\Local\Temp\ipykernel_8692\1835473429.py:4: UserWarning: The provided table name 'ORDER_ITEMS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_final_items[cols_to_load].to_sql(
